# tube_calc_v2 — corrected frame statics

Copy of `tube_calc.ipynb` with the review findings implemented. The original file is untouched.

| # | Original | Here |
|---|---|---|
| 1 | `r_p`, `r_hand` used as moment arms, with `sin(21.4deg)` patched onto one term only | every load is a **vector acting at a point**; arms come out of `r x F`, never hand-picked |
| 2 | `F_a = F_hand - F_b - P` contradicted the `F_hand + F_b + F_a + P = 0` written above it | fork solved as a 3x3 linear system, then `sum F = 0` and `sum M = 0` asserted |
| 3 | `F_b` was circular (`F_hand * r_hand` collapses back to `r_p * tempP`, so two different loads shared one arm) | each load has its own position vector |
| 4 | `D_x` and `F_net` were the same quantity with opposite signs; `F_st` had to be scraped out of one global force equation | no `D_x` at all: the rear triangle is in the model, so every load enters at a real node and both force equations are used |
| 5 | `alpha`, `beta` are fork-relative but `gamma`, `phi` are ground-relative, and `sin(beta + phi - gamma)` mixed them | one frame (`+x` forward, `+y` up). Member directions are derived from joint coordinates, so there is no angle convention left to mix |
| 6 | `np.abs(F)` + yield-only sizing; `od = 1.5 mm` printed as a design result | sign-aware: tension -> yield, compression -> min(yield, Euler, local shell buckling), then snapped to a stock tube OD |
| 7 | no way to know any of this was wrong | every free body prints its equilibrium residual and a test cell asserts them |

**Read the `ASSUMPTION` comments before trusting any number.** The frame coordinates and the load case are realistic placeholders; the point of this file is that the method is now checkable.

In [1]:
import numpy as np

MM = 1e-3
DEG = np.pi / 180

def pt(x_mm, y_mm):
    """A point: entered in mm, stored in metres."""
    return np.array([x_mm, y_mm], float) * MM

def unit(v):
    v = np.asarray(v, float)
    return v / np.linalg.norm(v)

def cross_z(r, f):
    """z component of r x f, i.e. the 2D moment of f about the origin."""
    return r[0] * f[1] - r[1] * f[0]

def polar(mag, ang_deg):
    a = ang_deg * DEG
    return np.array([mag * np.cos(a), mag * np.sin(a)])

def line_angle(v):
    """Angle of a member line, folded into [0, 180) deg from +x."""
    return np.degrees(np.arctan2(v[1], v[0])) % 180.0

def fmt(v, n=1):
    return "(" + ", ".join(f"{c:.{n}f}" for c in v) + ")"

print("helpers ready  |  frame: +x forward, +y up, +moment CCW  |  units: m, N, Pa, rad")

helpers ready  |  frame: +x forward, +y up, +moment CCW  |  units: m, N, Pa, rad


## 1. Inputs

Everything below is a measured frame dimension or a load-case decision. Angles are degrees **from `+x` (horizontal), CCW positive**.

In [2]:
PHI   = 66.0        # steering axis / head tube angle from horizontal, deg
GAMMA = 75.0        # seat tube angle from horizontal, deg

HT_TOP_XY    = (405.0, 585.0)   # top of head tube, mm from BB
GRIP_OFF     = (-60.0, 120.0)   # head tube top -> grip, mm
SEAT_LEN     = 560.0            # BB -> seat cluster along the seat tube, mm
SADDLE_OFF   = (-25.0, 75.0)    # seat cluster -> saddle, mm
REAR_AXLE_XY = (-420.0, -70.0)  # mm from BB
BB_DROP      = 70.0             # axles sit at y = -BB_DROP
BRG_SPACING  = 30.0             # mm, upper <-> lower headset bearing
CRANK_LEN    = 170.0            # mm
CRANK_ANG    = -21.4            # deg, driving crank: this is the angle the original
                                # notebook hid inside sin(21.4 * c)

F_GRIP   = 1200.0   # N, rider force on the bars, straight down
F_SADDLE = 2500.0   # N, rider weight plus dynamic factor on the saddle, down
F_CRANK  = 5000.0   # N, pedal force. KEPT FROM THE ORIGINAL NOTEBOOK.

# ASSUMPTION: all three rider loads are vertical. No braking force, no disc rotor
# reaction, no lateral load, no acceleration.
# ASSUMPTION: the crank force reaches the frame as a force through the BB spindle,
# so its line of action is set by the BB, not by the pedal position. The crank ANGLE
# therefore only matters through the drive torque, which is load case B below.
drive_torque = F_CRANK * CRANK_LEN * MM * np.cos(CRANK_ANG * DEG)
print(f"implied drive torque at the BB = {drive_torque:.0f} N.m")
print(f"  -> chain tension ~ {drive_torque/0.085:.0f} N on an 85 mm radius chainring")
print(f"  -> that is a second, larger load case for the chainstays and BB shell. NOT modelled here.")
print(f"original notebook: P = 5000 N (tube_calc.txt had 500000 N). A 5000 N pedal force")
print(f"is ~ 510 kgf, so this value still needs a source.")

implied drive torque at the BB = 791 N.m
  -> chain tension ~ 9311 N on an 85 mm radius chainring
  -> that is a second, larger load case for the chainstays and BB shell. NOT modelled here.
original notebook: P = 5000 N (tube_calc.txt had 500000 N). A 5000 N pedal force
is ~ 510 kgf, so this value still needs a source.


In [3]:
u_axis = polar(1.0, 180.0 - PHI)              # up the steering axis
n_axis = np.array([-u_axis[1], u_axis[0]])    # transverse to it

J3 = pt(0.0, 0.0)                                                  # bottom bracket
J1 = pt(*HT_TOP_XY)                                                # head tube top
J2 = SEAT_LEN * MM * np.array([-np.cos(GAMMA*DEG), np.sin(GAMMA*DEG)])   # seat cluster
R  = pt(*REAR_AXLE_XY)                                             # rear axle
GRIP   = J1 + pt(*GRIP_OFF)
SADDLE = J2 + pt(*SADDLE_OFF)
BRG_UP = J1
BRG_LO = J1 - BRG_SPACING * MM * u_axis
t_axle = (-BB_DROP * MM - J1[1]) / u_axis[1]
FRONT = J1 + t_axle * u_axis                                       # front axle, on the axis

NODES = {"J1 head tube top": J1, "J2 seat cluster": J2,
         "J3 bottom bracket": J3, "R rear axle": R}
MEMBERS = {"TT": (J1, J2), "DT": (J1, J3), "ST": (J2, J3), "CS": (J3, R), "SS": (J2, R)}

print(f"front axle {fmt(FRONT/MM, 1)} mm    wheelbase {(FRONT[0]-R[0])/MM:.1f} mm")
print(f"{'member':6} {'len mm':>8} {'deg from +x':>12} {'acute to horiz':>15} {'deg to fork axis':>18}")
for name, (a, b) in MEMBERS.items():
    d = b - a
    la = line_angle(d)
    to_ax = abs(la - (180.0 - PHI)) % 180.0
    to_ax = min(to_ax, 180.0 - to_ax)
    print(f"{name:6} {np.linalg.norm(d)/MM:8.1f} {np.degrees(np.arctan2(d[1],d[0])):12.1f} "
          f"{min(la, 180.0-la):15.1f} {to_ax:18.1f}")

front axle (696.6, -70.0) mm    wheelbase 1116.6 mm
member   len mm  deg from +x  acute to horiz   deg to fork axis
TT        551.7       -175.4             4.6               70.6
DT        711.5       -124.7            55.3               58.7
ST        560.0        -75.0            75.0                9.0
CS        425.8       -170.5             9.5               75.5
SS        670.0       -114.2            65.8               48.2


### Cross-check against the angles you measured

The whiteboard lists `alpha` and `beta` as **"angle of DT to fork"** and **"angle of TT to fork"**, while `gamma` and `phi` are ground-referenced. That mixed convention is what produced `sin(beta + phi - gamma)`. Here the member directions come out of coordinates, so these angles are **outputs** — and comparing them with your measured values becomes a real geometry check instead of a silent unit of account.

In [4]:
ALPHA_ORIG = 62.6   # notebook: "angle of DT to fork"
BETA_ORIG  = 91.1   # notebook: "angle of TT to fork"

axis_line = (180.0 - PHI) % 180.0
def to_axis(a, b):
    la = line_angle(b - a)
    d = abs(la - axis_line) % 180.0
    return min(d, 180.0 - d)

st_line = line_angle(J3 - J2)
print(f"seat tube from horizontal : {min(st_line, 180-st_line):5.1f} deg  vs GAMMA = {GAMMA:5.1f}  -> agrees by construction")
print(f"DT to fork axis           : {to_axis(J1, J3):5.1f} deg  vs ALPHA_ORIG = {ALPHA_ORIG:5.1f}  diff {abs(to_axis(J1,J3)-ALPHA_ORIG):4.1f}")
print(f"TT to fork axis           : {to_axis(J1, J2):5.1f} deg  vs BETA_ORIG  = {BETA_ORIG:5.1f}  diff {abs(to_axis(J1,J2)-BETA_ORIG):4.1f}")
print(f"\nangle between the seat tube and the steering axis = {to_axis(J2, J3):.1f} deg = GAMMA - PHI = {GAMMA-PHI:.1f} deg")
print("That is the quantity the original COMMENT called sin(gamma - phi), and that the")
print("original CODE replaced with sin(beta + phi - gamma) = sin(82.1 deg).")
print(f"  sin( 9.0deg) = {np.sin((GAMMA-PHI)*DEG):.4f}")
print(f"  sin(82.1deg) = {np.sin((BETA_ORIG+PHI-GAMMA)*DEG):.4f}")
print(f"  ratio        = {np.sin((BETA_ORIG+PHI-GAMMA)*DEG)/np.sin((GAMMA-PHI)*DEG):.2f}x  -> F_st was ~6x too small")

seat tube from horizontal :  75.0 deg  vs GAMMA =  75.0  -> agrees by construction
DT to fork axis           :  58.7 deg  vs ALPHA_ORIG =  62.6  diff  3.9
TT to fork axis           :  70.6 deg  vs BETA_ORIG  =  91.1  diff 20.5

angle between the seat tube and the steering axis = 9.0 deg = GAMMA - PHI = 9.0 deg
That is the quantity the original COMMENT called sin(gamma - phi), and that the
original CODE replaced with sin(beta + phi - gamma) = sin(82.1 deg).
  sin( 9.0deg) = 0.1564
  sin(82.1deg) = 0.9905
  ratio        = 6.33x  -> F_st was ~6x too small


## 2. Whole-bike equilibrium

Supports are the two tyre contact patches, taken vertical. Loads are the rider at the bars, the saddle and the cranks.

In [5]:
def solve_two_axle(loads, r_rear, r_front):
    """loads = [(point, force)]. Returns R_rear, R_front (vertical) and the residuals."""
    fy = sum(f[1] for _, f in loads)
    m_rear = sum(cross_z(p - r_rear, f) for p, f in loads)
    arm = r_front[0] - r_rear[0]
    rf = -m_rear / arm
    rr = -fy - rf
    return rr, rf, rr + rf + fy, m_rear + arm * rf

APPLIED = [
    (GRIP,  np.array([0.0, -F_GRIP])),
    (SADDLE, np.array([0.0, -F_SADDLE])),
    (J3,    np.array([0.0, -F_CRANK])),
]
R_REAR, R_FRONT, RES_G_F, RES_G_M = solve_two_axle(APPLIED, R, FRONT)
print(f"R_front = {R_FRONT:8.1f} N up at {fmt(FRONT/MM,1)}")
print(f"R_rear  = {R_REAR:8.1f} N up at {fmt(R/MM,1)}")
print(f"equilibrium residual: sum F = {RES_G_F:.2e} N, sum M = {RES_G_M:.2e} N.m")

R_front =   3262.6 N up at (696.6, -70.0)
R_rear  =   5437.4 N up at (-420.0, -70.0)
equilibrium residual: sum F = 0.00e+00 N, sum M = 0.00e+00 N.m


## 3. Fork free body

Forces on the fork: the grip load, the front tyre reaction, and the two headset bearings. That is four unknown components for three equations, so one assumption is needed — **the upper bearing takes no thrust along the steering axis**, the lower one takes it all. This is the standard headset idealisation, and it is what the `M_A => F_B` / `F => F_A` sketch on the board was doing.

The fork hands the frame a force **and a couple** at the head tube. The original notebook dropped the couple silently.

In [6]:
def solve_fork(r_grip, f_grip, r_axle, f_axle, r_up, r_lo, u, n):
    """Unknowns: lower bearing axial, lower bearing transverse, upper bearing transverse."""
    f_ext = f_grip + f_axle
    A = np.array([[u[0], n[0], n[0]],
                  [u[1], n[1], n[1]],
                  [0.0, 0.0, cross_z(r_up - r_lo, n)]])
    b = np.array([-f_ext[0], -f_ext[1],
                  -(cross_z(r_grip - r_lo, f_grip) + cross_z(r_axle - r_lo, f_axle))])
    x = np.linalg.solve(A, b)
    return x[0] * u + x[1] * n, x[2] * n

F_LO, F_UP = solve_fork(GRIP, APPLIED[0][1], FRONT, np.array([0.0, R_FRONT]),
                        BRG_UP, BRG_LO, u_axis, n_axis)
f_ext_fork = APPLIED[0][1] + np.array([0.0, R_FRONT])
RES_FK_F = F_LO + F_UP + f_ext_fork
RES_FK_M = (cross_z(GRIP - BRG_LO, APPLIED[0][1]) + cross_z(FRONT - BRG_LO, np.array([0.0, R_FRONT]))
            + cross_z(BRG_UP - BRG_LO, F_UP))

F_NET = f_ext_fork                                    # force the fork puts into the frame
M_HT = cross_z(GRIP - J1, APPLIED[0][1]) + cross_z(FRONT - J1, np.array([0.0, R_FRONT]))

print(f"lower bearing {fmt(F_LO,1)} N  |{np.linalg.norm(F_LO):8.1f}|")
print(f"upper bearing {fmt(F_UP,1)} N  |{np.linalg.norm(F_UP):8.1f}|")
print(f"fork residual: sum F = {np.linalg.norm(RES_FK_F):.2e} N, sum M = {RES_FK_M:.2e} N.m")
print(f"\nresultant into the frame at the head tube: {fmt(F_NET,1)} N")
print(f"couple into the head tube:                 {M_HT:+.1f} N.m   <- dropped by the original model")
print(f"bearing loads are ~{np.linalg.norm(F_UP)/max(1.0,np.linalg.norm(f_ext_fork)):.0f}x the net fork load because the bearing")
print(f"spacing is only {BRG_SPACING:.0f} mm. That is real, and it is why headsets and head tubes are overbuilt.")

lower bearing (-30399.8, -15597.5) N  | 34167.6|
upper bearing (30399.8, 13534.8) N  | 33276.7|
fork residual: sum F = 3.75e-12 N, sum M = 0.00e+00 N.m

resultant into the frame at the head tube: (0.0, 2062.6) N
couple into the head tube:                 +1023.5 N.m   <- dropped by the original model
bearing loads are ~16x the net fork load because the bearing
spacing is only 30 mm. That is real, and it is why headsets and head tubes are overbuilt.


## 4. Frame as a pin-jointed truss

Five members, four nodes, and every load enters at the node where it physically arrives. Two couples remain that a pin-jointed truss cannot carry — the head tube couple, and the couple created by moving the saddle load onto the seat cluster node. Each is replaced by a **self-equilibrated vertical force pair across the two nodes it acts between**, so the nodal force system satisfies `sum F = 0` *and* `sum M = 0` and the truss actually has a solution.

That pair is an idealisation of a real mechanism (the TT and DT bosses sit at different heights on the head tube; the saddle sits behind the seat tube axis), not a bending analysis. The original notebook had no rear triangle and no saddle load, which is exactly why `F_st` could not come out of a joint.

In [7]:
def couple_to_pair(r_a, r_b, moment, d=np.array([0.0, 1.0])):
    """(+V*d at r_a, -V*d at r_b) equivalent to a pure couple."""
    v = moment / cross_z(r_a - r_b, d)
    return v * d, -v * d

m_saddle = cross_z(SADDLE - J2, APPLIED[1][1])   # lost by moving the saddle load to J2
pa1, pb1 = couple_to_pair(J1, J3, M_HT)          # head tube couple, smeared J1 <-> J3
pa2, pb2 = couple_to_pair(J2, J3, m_saddle)      # saddle offset couple, smeared J2 <-> J3

NODE_LOAD = {
    "J1 head tube top":  F_NET + pa1,
    "J2 seat cluster":   APPLIED[1][1] + pa2,
    "J3 bottom bracket": APPLIED[2][1] + pb1 + pb2,
    "R rear axle":       np.array([0.0, R_REAR]),
}
RES_N_F = sum(NODE_LOAD.values())
RES_N_M = sum(cross_z(NODES[k], v) for k, v in NODE_LOAD.items())

print(f"{'node':20} {'load N':>22}")
for name, f in NODE_LOAD.items():
    print(f"{name:20} {fmt(f,1):>22}")
print(f"\nclosure: sum F = {fmt(RES_N_F, 4)} N   sum M about BB = {RES_N_M:.3e} N.m")
print(f"head tube couple {M_HT:+.1f} N.m -> pair {pa1[1]:+.1f} / {pb1[1]:+.1f} N vertical")
print(f"saddle offset couple {m_saddle:+.1f} N.m -> pair {pa2[1]:+.1f} / {pb2[1]:+.1f} N vertical")

node                                 load N
J1 head tube top              (0.0, 4589.7)
J2 seat cluster              (0.0, -2931.2)
J3 bottom bracket            (0.0, -7095.9)
R rear axle                   (0.0, 5437.4)

closure: sum F = (0.0000, 0.0000) N   sum M about BB = 9.095e-13 N.m
head tube couple +1023.5 N.m -> pair +2527.1 / -2527.1 N vertical
saddle offset couple +62.5 N.m -> pair -431.2 / +431.2 N vertical


In [8]:
def solve_truss(nodes, members, node_load):
    """Axial force per member, tension positive. Least squares over every joint equation."""
    keys = list(nodes)
    A = np.zeros((2 * len(keys), len(members)))
    b = np.zeros(2 * len(keys))
    for i, kn in enumerate(keys):
        p = nodes[kn]
        for j, (name, (s, e)) in enumerate(members.items()):
            if np.allclose(p, s):
                u = unit(e - s)
            elif np.allclose(p, e):
                u = unit(s - e)
            else:
                continue
            A[2 * i, j] = u[0]
            A[2 * i + 1, j] = u[1]
        f = node_load[kn]
        b[2 * i], b[2 * i + 1] = -f[0], -f[1]
    x, _, rank, _ = np.linalg.lstsq(A, b, rcond=None)
    return x, np.max(np.abs(A @ x - b)), rank, A.shape

F_MEM, TRUSS_RESID, TRUSS_RANK, TRUSS_SHAPE = solve_truss(NODES, MEMBERS, NODE_LOAD)
print(f"{'member':8} {'axial force N':>14}  {'state':>12}")
for name, val in zip(MEMBERS, F_MEM):
    print(f"{name:8} {val:14.1f}  {'tension' if val > 0 else 'compression':>12}")
print(f"\njoint-equation residual = {TRUSS_RESID:.2e} N over {TRUSS_SHAPE[0]} equations, "
      f"{len(MEMBERS)} unknowns, rank {TRUSS_RANK}")
print("The original notebook got F_dt = -135 N, i.e. a down tube carrying nothing.")
print("Under a pedal load the down tube is one of the most heavily loaded tubes on a frame.")

member    axial force N         state
TT              -3375.0   compression
DT               5910.3       tension
ST               2772.0       tension
CS               2683.2       tension
SS              -6446.8   compression

joint-equation residual = 5.46e-12 N over 8 equations, 5 unknowns, rank 5
The original notebook got F_dt = -135 N, i.e. a down tube carrying nothing.
Under a pedal load the down tube is one of the most heavily loaded tubes on a frame.


## 5. Tube sizing

`fail = 290000000` was unlabeled. The whiteboard says *tensile yield strength* of 6061, so this uses **6061-T6 yield = 276 MPa** and says so. Compression members are sized on whichever of yield / Euler / local shell buckling governs, and the answer is snapped to a stock tube OD instead of printing a 1.5 mm solid rod.

In [9]:
E_ALU  = 69e9      # Pa, 6061
SIG_Y  = 276e6     # Pa, 6061-T6 yield (the board says "tensile yield strength")
FOS    = 2.0
WALL   = 0.75e-3   # m
STD_OD = np.array([22.2, 25.4, 27.2, 28.6, 31.8, 34.9, 38.1, 42.2, 50.8, 54.0, 57.2, 60.3]) * MM

def area_ann(od, t):
    return np.pi * t * (od - t) if od > 2 * t else 0.0    # exact: pi/4*(od^2 - id^2)

def inertia(od, t):
    return np.pi / 64 * (od**4 - (od - 2*t)**4) if od > 2 * t else 0.0

def sig_euler(od, t, L, k=1.0):
    a = area_ann(od, t)
    return np.pi**2 * E_ALU * inertia(od, t) / (k * L)**2 / a if a > 0 else 0.0

def sig_shell(od, t):
    """Classical local buckling of a short isotropic cylinder under uniform compression."""
    return 0.605 * E_ALU * t / ((od - t) / 2.0)

def size_member(force, L, name=""):
    mag = abs(force)
    state = "tension" if force >= 0 else "compression"
    out = {"name": name, "force": force, "L": L, "state": state,
           "od_yield_only": mag / (np.pi * WALL * (SIG_Y / FOS)) + WALL,
           "od": None, "governs": None, "util": None}
    if state == "tension":
        allow, mode = SIG_Y / FOS, "yield"
        for od in STD_OD:
            if mag / area_ann(od, WALL) <= allow:
                out["od"], out["governs"], out["util"] = od, mode, mag / area_ann(od, WALL) / allow
                break
        return out
    for od in STD_OD:
        cand = {"yield": SIG_Y / FOS, "Euler": sig_euler(od, WALL, L) / FOS,
                "local shell": sig_shell(od, WALL) / FOS}
        mode = min(cand, key=cand.get)
        if mag / area_ann(od, WALL) <= cand[mode]:
            out["od"], out["governs"], out["util"] = od, mode, mag / area_ann(od, WALL) / cand[mode]
            break
    return out

SIZES = [size_member(val, np.linalg.norm(b - a), name)
         for name, (a, b), val in zip(MEMBERS, MEMBERS.values(), F_MEM)]

print(f"{'tube':5} {'force N':>10} {'state':>12} {'yield-only OD mm':>19} {'stock OD mm':>12} {'governs':>12} {'use':>6}")
for s in SIZES:
    od = f"{s['od']/MM:.1f}" if s["od"] is not None else "NONE in stock"
    util = f"{s['util']:.2f}" if s["util"] is not None else "  -"
    print(f"{s['name']:5} {s['force']:10.1f} {s['state']:>12} {s['od_yield_only']/MM:19.2f} "
          f"{od:>12} {str(s['governs']):>12} {util:>6}")
print("\nfor reference, production aluminium frames: seat tube 27.2-28.6, down tube 31.8-50.8,")
print("top tube 25.4-38.1 mm OD. The wall here is fixed at 0.75 mm, which no real frame uses")
print("on a 25 mm tube carrying these loads.")

tube     force N        state    yield-only OD mm  stock OD mm      governs    use
TT       -3375.0  compression               11.13         25.4        Euler   0.68
DT        5910.3      tension               18.93         22.2        yield   0.85
ST        2772.0      tension                9.28         22.2        yield   0.40
CS        2683.2      tension                9.00         22.2        yield   0.38
SS       -6446.8  compression               20.58         31.8        Euler   0.96

for reference, production aluminium frames: seat tube 27.2-28.6, down tube 31.8-50.8,
top tube 25.4-38.1 mm OD. The wall here is fixed at 0.75 mm, which no real frame uses
on a 25 mm tube carrying these loads.


## 6. Tests

Each check is an assertion on a residual, so a wrong sign or a wrong arm fails loudly instead of printing a plausible-looking number.

In [10]:
TESTS = []
def check(label, value, tol=1e-6):
    ok = value <= tol
    TESTS.append((label, value, ok))
    print(f"{'PASS' if ok else 'FAIL'}  {label:62} {value:.3e} (tol {tol:.0e})")

def check_eq(label, got, want, tol=1e-9):
    check(label, abs(got - want) / max(1.0, abs(want)), tol)

def check_big(label, value, floor):
    """A regression test: this quantity is supposed to be large, because it is a bug."""
    ok = value >= floor
    TESTS.append((label, value, ok))
    print(f"{'PASS' if ok else 'FAIL'}  {label:62} {value:.3e} (want >= {floor:.0e})")

scale_f = max(1.0, np.max(np.abs(F_MEM)))
scale_n = max(F_GRIP, F_SADDLE, F_CRANK, abs(R_REAR), abs(R_FRONT))
check("T1a global sum F", abs(RES_G_F) / scale_n, 1e-9)
check("T1b global sum M", abs(RES_G_M) / (scale_n * max(1.0, FRONT[0] - R[0])), 1e-9)
check("T2a fork sum F", np.linalg.norm(RES_FK_F) / max(1.0, np.linalg.norm(F_LO)), 1e-9)
check("T2b fork sum M", abs(RES_FK_M) / max(1.0, abs(M_HT)), 1e-9)
check("T3a nodal load sum F", np.linalg.norm(RES_N_F) / scale_n, 1e-9)
check("T3b nodal load sum M", abs(RES_N_M) / (scale_n * max(1.0, abs(J1[0]), abs(R[0]))), 1e-9)
check("T4 truss joint-equation residual", TRUSS_RESID / scale_f, 1e-9)
check_eq("T4 truss rank equals member count", TRUSS_RANK, len(MEMBERS), 0.0)

# T5 known answer: 100 N hanging load, two members at +-45 deg -> 70.711 N each
kn = {"L": np.array([0.0, 0.0])}
km = {"m1": (kn["L"], np.array([1.0, 1.0])), "m2": (kn["L"], np.array([-1.0, 1.0]))}
kx, kres, _, _ = solve_truss(kn, km, {"L": np.array([0.0, -100.0])})
check_eq("T5 known-answer truss, member 1", kx[0], 100 / np.sqrt(2), 1e-12)
check_eq("T5 known-answer truss, member 2", kx[1], 100 / np.sqrt(2), 1e-12)
check("T5 known-answer truss residual", kres, 1e-12)

# T6 linearity: 3x every load -> 3x every member force
f3, r3, _, _ = solve_truss(NODES, MEMBERS, {k: v * 3.0 for k, v in NODE_LOAD.items()})
check("T6 linearity under 3x load", np.max(np.abs(f3 - 3 * F_MEM)) / scale_f, 1e-9)

# T7 sizing: the chosen section really has the area it needs, and the annulus
#    identity used for id = A/(pi t) - t is exact, not a thin-wall approximation
for s in SIZES:
    if s["od"] is not None:
        a_req = abs(s["force"]) / (SIG_Y / FOS)
        a_got = area_ann(s["od"], WALL)
        check(f"T7 {s['name']} stock section >= yield-only requirement", max(0.0, a_req - a_got) / a_req, 0.0)
check_eq("T7 annulus identity A = pi*t*(od-t)",
         area_ann(0.0318, WALL), np.pi / 4 * (0.0318**2 - (0.0318 - 2*WALL)**2), 1e-12)

print("\n--- T8 regression: the ORIGINAL notebook arithmetic, rebuilt exactly ---")
P0, rp0, rh0, rb0 = 5000.0, 672.08e-3, 750.633e-3, 30e-3
al0, be0, ga0, ph0 = 62.6*DEG, 91.1*DEG, 75*DEG, 66*DEG
tempP0 = P0 * np.sin(21.4 * DEG)
Fhand0 = rp0 * tempP0 / rh0
Dx0 = Fhand0 - P0
Fb0 = (Fhand0 * rh0 + P0 * rp0) / rb0
Fa0 = Fhand0 - Fb0 - P0
Fnet0 = -(Fa0 + Fb0)
Ftt0 = Fnet0 * np.cos(al0) / np.sin(al0 - be0)
Fdt0 = -Ftt0 * np.cos(be0) / np.cos(al0)
Fst0 = (Dx0 + Fdt0 * np.sin(al0) + Ftt0 * np.sin(be0)) / np.sin(be0 + ph0 - ga0)
print(f"reproduced: F_hand {Fhand0:.5f}  F_a {Fa0:.2f}  F_b {Fb0:.2f}  "
      f"F_tt {Ftt0:.2f}  F_dt {Fdt0:.2f}  F_st {Fst0:.2f}")
check_eq("T8a reproduces the original notebook output", Fst0, -6797.584826265671, 1e-9)
check_big("T8b BUG original fork FBD sum F is not zero", abs(Fhand0 + Fb0 + Fa0 + P0) / P0, 0.2)
check("T8c BUG F_net is just -D_x, so the fork couple is double counted", abs(Fnet0 + Dx0) / P0, 1e-9)
check("T8d BUG F_b is circular: F_hand*r_hand collapses to r_p*tempP", abs(Fhand0*rh0 - rp0*tempP0) / (rp0*P0), 1e-9)
check_eq("T8e BUG F_st denominator off by sin(b+p-g)/sin(g-p)",
         np.sin(be0 + ph0 - ga0) / np.sin(ga0 - ph0), 6.3302, 1e-3)
print(f"    -> T8b: the fork FBD is out of balance by {Fhand0 + Fb0 + Fa0 + P0:+.1f} N")
print(f"    -> T8c: F_net = {Fnet0:.2f} and D_x = {Dx0:.2f}, the same number with the sign flipped")
print(f"    -> T8e: F_st should be {Fst0*6.3302:.0f} N by the code's own comment, not {Fst0:.0f} N")
print(f"    -> T8d: F_b = r_p*(tempP + P)/r_b, so P and F_hand were given the same moment arm")

n_pass = sum(1 for t in TESTS if t[2])
print(f"\n{n_pass}/{len(TESTS)} checks passed")
assert all(t[2] for t in TESTS), "a check failed"
print("ALL CHECKS PASSED   (T8b-T8e are meant to pass: they assert the original bugs are real)")

PASS  T1a global sum F                                               0.000e+00 (tol 1e-09)
PASS  T1b global sum M                                               0.000e+00 (tol 1e-09)
PASS  T2a fork sum F                                                 1.098e-16 (tol 1e-09)
PASS  T2b fork sum M                                                 0.000e+00 (tol 1e-09)
PASS  T3a nodal load sum F                                           1.673e-16 (tol 1e-09)
PASS  T3b nodal load sum M                                           1.673e-16 (tol 1e-09)
PASS  T4 truss joint-equation residual                               8.465e-16 (tol 1e-09)
PASS  T4 truss rank equals member count                              0.000e+00 (tol 0e+00)
PASS  T5 known-answer truss, member 1                                0.000e+00 (tol 1e-12)
PASS  T5 known-answer truss, member 2                                4.019e-16 (tol 1e-12)
PASS  T5 known-answer truss residual                                 1.974e-14 (tol 1e-12)

## 7. Results

In [11]:
print(f"{'tube':5} {'axial force N':>14} {'state':>12} {'stock OD x wall':>16} {'governs':>12} {'utilisation':>12}")
for s in SIZES:
    if s["od"] is None:
        print(f"{s['name']:5} {s['force']:14.1f} {s['state']:>12} {'NONE in stock':>16}")
    else:
        label = f"{s['od']/MM:.1f} x {WALL/MM:.2f}"
        print(f"{s['name']:5} {s['force']:14.1f} {s['state']:>12} {label:>16} "
              f"{s['governs']:>12} {s['util']:12.2f}")

orig = {"TT": -3246.88, "DT": -135.45, "ST": -6797.58}
print("\nsame three members, original notebook vs here:")
for s in SIZES:
    if s["name"] in orig:
        print(f"  {s['name']}: {orig[s['name']]:10.1f} N  ->  {s['force']:10.1f} N")
print("\nThe original numbers cannot be compared directly: they came from a load case whose")
print("free bodies did not close, so there is no single scale factor between the two.")

ss_up = F_MEM[list(MEMBERS).index("SS")] * unit(R - J2)[1]
print(f"\nWhy the seat tube comes out in tension here: the seatstay pushes the cluster up with")
print(f"{ss_up:+.0f} N while the saddle load pushes it down with {NODE_LOAD['J2 seat cluster'][1]:+.0f} N.")
print("Change the split between saddle load and rear tyre reaction and the sign flips, so the")
print("seat tube is the member most sensitive to how this load case is defined.")

tube   axial force N        state  stock OD x wall      governs  utilisation
TT           -3375.0  compression      25.4 x 0.75        Euler         0.68
DT            5910.3      tension      22.2 x 0.75        yield         0.85
ST            2772.0      tension      22.2 x 0.75        yield         0.40
CS            2683.2      tension      22.2 x 0.75        yield         0.38
SS           -6446.8  compression      31.8 x 0.75        Euler         0.96

same three members, original notebook vs here:
  TT:    -3246.9 N  ->     -3375.0 N
  DT:     -135.4 N  ->      5910.3 N
  ST:    -6797.6 N  ->      2772.0 N

The original numbers cannot be compared directly: they came from a load case whose
free bodies did not close, so there is no single scale factor between the two.

Why the seat tube comes out in tension here: the seatstay pushes the cluster up with
+5878 N while the saddle load pushes it down with -2931 N.
Change the split between saddle load and rear tyre reaction and the sig

### Still not modelled

These are the reasons the table above is a first pass and not a frame:

- **Drive torque.** `F_CRANK` at a 170 mm crank is `~790 N.m`, which is `~9.3 kN` of chain tension. That is the dominant chainstay, seatstay and BB shell load case and it cannot be represented by a pin-jointed truss.
- **Bending.** Every member here is a two-force member. The head tube couple printed in step 3 is exactly the load that sizes a head tube, and it is smeared into a force pair rather than solved.
- **Braking and the disc rotor reaction**, a second and often larger case for the fork, head tube and seat tube cluster.
- **Fatigue.** Frames are signed off on ~100k cycles at 1.2-2x working load (ISO 4210), not on a 2x static yield margin. A 0.75 mm wall at a welded joint is fatigue-governed, not net-section-governed.
- **Local buckling** uses the classical `0.605 E t / r` shell formula with no knock-down for a weld seam or geometric imperfection; real knock-down is 0.5-0.7.
- **The geometry inputs are placeholders.** Replace `HT_TOP_XY`, `SEAT_LEN`, `REAR_AXLE_XY`, `GRIP_OFF`, `SADDLE_OFF` and all three loads with measured values before using any output. The `T8a` test will keep reproducing the old numbers regardless, so it is also the diff against `tube_calc.ipynb`.